# DEMO 2 — `brz_01_arancione_sales` with a `StepLog` recorder object (cells intact)

**Illustrative only.** Placeholder names; not meant to run. Delete after reading.

This is the alternative to the context-manager version (`NEW_STEP_LOG.ipynb`).
A plain **recorder object** keeps the original cell-by-cell structure — you can
still run validate, read, and write as separate cells and inspect between them —
while removing the ~12-line `except` block and the §11.4 pre-declared-variable
footgun.

The deal:
- `StepLog(...)` is constructed once (writes the `RUNNING` row) and held in `step`.
- Each work cell keeps a tiny `try/except Exception as e: step.fail(e); raise`
  (2 lines, not 12). `step.fail()` captures the traceback and writes `FAILED`.
- You set `step.rows_read` / `step.rows_written` on the object.
- The success close is **explicit**: `step.succeed()` in the write cell.
- Leaf writes (`ingestion_log_insert`) return a dict and never raise (Decision 4).

## What `StepLog` is (sketch — lives ONCE in `pipeline_logging.py`)

```python
class StepLog:
    def __init__(self, spark, audit_schema, dbutils, *, pipeline_run_id,
                 step_sequence, notebook_folder, notebook_name,
                 layer=None, target_table=None):
        self._spark   = spark
        self._audit   = audit_schema
        self.step_log_id = str(uuid.uuid4())
        self._started = datetime.now(timezone.utc)
        self.rows_read = 0
        self.rows_written = 0
        self._common = dict(
            pipeline_run_id=pipeline_run_id, step_sequence=step_sequence,
            notebook_folder=notebook_folder, notebook_name=notebook_name,
            layer=layer, target_table=target_table)
        # OPEN: write the RUNNING row (was the old "cell 3")
        pipeline_step_log_upsert(spark, audit_schema, self.step_log_id,
                                 status=STATUS_RUNNING,
                                 started_timestamp=self._started, **self._common)

    def _close(self, status, error_message=None):
        pipeline_step_log_upsert(
            self._spark, self._audit, self.step_log_id, status=status,
            started_timestamp=self._started,
            ended_timestamp=datetime.now(timezone.utc),
            rows_read=self.rows_read, rows_written=self.rows_written,
            error_message=error_message, **self._common)

    def succeed(self):  self._close(STATUS_SUCCEEDED)
    def no_files(self): self._close(STATUS_NO_FILES)
    def fail(self, exc):
        err = Utils.capture_exception(exc)
        self._close(STATUS_FAILED,
                    error_message=f"{err['error_type']}: {err['error_message']}")
```

Because OPEN and every close use the same `step_log_id`, each close is a MERGE
update on that key — the same two-call upsert the original did by hand, but the
12-line handler now lives here once instead of in every cell.

In [ ]:
%run "../../libs/notebook_init"
# (illustrative path) injects CATALOG, BRONZE, AUDIT, RAW_FILES, STATUS_*,
# PIPELINE_RUN_ID, Utils, F, datetime, uuid, StepLog + audit helpers.

In [ ]:
# =============================================================================
# Cell 2 — Constants  (UNCHANGED from the original)
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

STORE_NAME      = "Arancione"
SOURCE_SUBPATH  = "arancione/"
SOURCE_PATH     = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE    = f"{BRONZE}.sales_arancione"
SOURCE_ENCODING = "windows-1252"

EXPECTED_SOURCE_COLS = [
    "OnlineRetailer", "SalesMonth", "Title", "Vintage",
    "Variety", "Score", "ListPrice", "Quantity",
]
HASH_SOURCE_COLS = [
    "online_retailer", "sales_month", "title", "vintage",
    "variety", "score", "list_price", "quantity",
]
read_schema = StructType([
    StructField("OnlineRetailer", StringType(), True),
    StructField("SalesMonth",     StringType(), True),
    StructField("Title",          StringType(), True),
    StructField("Vintage",        StringType(), True),
    StructField("Variety",        StringType(), True),
    StructField("Score",          StringType(), True),
    StructField("ListPrice",      StringType(), True),
    StructField("Quantity",       StringType(), True),
])

In [ ]:
# =============================================================================
# Cell 3 — Open the step log (writes the RUNNING row)
# Replaces the original cell 3 + all the loose status/ended_timestamp/
# error_message/rows_* variables. State now lives on `step`.
# =============================================================================
nb = Utils.get_notebook_context(dbutils)

step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = 1,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)

In [ ]:
# =============================================================================
# Cell 4 — File validation  (runnable on its own)
# =============================================================================
try:
    files = [f.path for f in dbutils.fs.ls(SOURCE_PATH)
             if f.path.lower().endswith(".csv")]

    if not files:
        step.no_files()                       # writes NO_FILES terminal row
        dbutils.notebook.exit("No CSV files found at " + SOURCE_PATH)

    bad_files = []
    for file_path in files:
        actual = (spark.read.format("csv").option("header", "true")
                  .option("encoding", SOURCE_ENCODING)
                  .load(file_path).limit(0).columns)
        if actual != EXPECTED_SOURCE_COLS:
            bad_files.append((file_path, actual))
    if bad_files:
        raise ValueError(f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).")

    print(f"[{TARGET_TABLE}] {len(files)} file(s) validated")

except dbutils.NotebookExit:
    raise                                     # clean exit — not a failure
except Exception as e:
    step.fail(e); raise                       # the whole handler, in 2 lines

In [ ]:
# =============================================================================
# Cell 5 — Read, shape, compute row_hash  (runnable on its own; inspect bronze_df after)
# =============================================================================
try:
    raw_df = (spark.read.format("csv").option("header", "true")
              .option("delimiter", ",").option("encoding", SOURCE_ENCODING)
              .schema(read_schema).load(SOURCE_PATH)
              .withColumn("source_file_path", F.col("_metadata.file_path")))

    bronze_df = (
        raw_df.select(
            F.col("OnlineRetailer").alias("online_retailer"),
            F.col("SalesMonth").alias("sales_month"),
            F.col("Title").alias("title"),
            F.col("Vintage").alias("vintage"),
            F.col("Variety").alias("variety"),
            F.col("Score").alias("score"),
            F.col("ListPrice").alias("list_price"),
            F.col("Quantity").alias("quantity"),
            F.col("source_file_path"),
        )
        .withColumn("row_hash", F.md5(F.concat_ws("|", *[F.col(c) for c in HASH_SOURCE_COLS])))
        .withColumn("inserted_ts", F.current_timestamp())
        .withColumn("run_id",      F.lit(PIPELINE_RUN_ID))
        .withColumn("store_name",  F.lit(STORE_NAME))
    )

    step.rows_read = bronze_df.count()        # set on the recorder
    print(f"[{TARGET_TABLE}] Prepared {step.rows_read:,} rows")

except Exception as e:
    step.fail(e); raise

In [ ]:
# =============================================================================
# Cell 6 — MERGE write (idempotent on row_hash)  (runnable on its own)
# step.succeed() is the explicit success close.
# =============================================================================
try:
    pre_count = spark.table(TARGET_TABLE).count()

    bronze_df.createOrReplaceTempView("bronze_staging")
    spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS target
        USING bronze_staging AS source
        ON target.row_hash = source.row_hash
        WHEN NOT MATCHED THEN INSERT *
    """)

    post_count = spark.table(TARGET_TABLE).count()
    step.rows_written = post_count - pre_count
    rows_skipped = step.rows_read - step.rows_written

    if step.rows_written + rows_skipped != step.rows_read:
        raise AssertionError(f"[{TARGET_TABLE}] Row count inconsistency.")

    print(f"[{TARGET_TABLE}] inserted={step.rows_written:,} skipped={rows_skipped:,}")

    step.succeed()                            # <-- explicit SUCCEEDED close

except Exception as e:
    step.fail(e); raise

In [ ]:
# =============================================================================
# Cell 7 — Ingestion log (leaf write)
# Decision 4: returns a dict, never raises. The Bronze write already stands
# (step.succeed() ran in cell 6), so a logging hiccup only warns.
# =============================================================================
res = ingestion_log_insert(
    spark, AUDIT,
    bronze_df.select("source_file_path").distinct(),
    PIPELINE_RUN_ID, step.step_log_id, STORE_NAME, TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"[{TARGET_TABLE}] WARNING: ingestion_log failed, Bronze write stands. {res['error_message']}")

In [ ]:
# =============================================================================
# Cell 8 — Archive processed files (post-success side effect)
# =============================================================================
try:
    Utils.move_all_files(
        dbutils, source_path=SOURCE_PATH,
        target_path=f"{SOURCE_PATH}archive", create_target=True, skip_dirs=True,
    )
except Exception as e:
    # step already SUCCEEDED; surface the archive failure to the orchestrator.
    print(f"[{SOURCE_PATH}archive] WARNING: archive failed — Bronze write stands.")
    raise

## How this compares

| Aspect | Original | Context manager (`NEW_STEP_LOG`) | Recorder object (this file) |
|---|---|---|---|
| Cells stay separately runnable | yes | **no** (one big cell) | **yes** |
| `except` boilerplate per work cell | ~12 lines | 0 | **2 lines** (`step.fail(e); raise`) |
| §11.4 pre-declared-var footgun | present | gone | **gone** (state on `step`) |
| Where the 12-line handler lives | every cell | once (`step_log`) | **once (`StepLog`)** |
| Success close | manual upsert | automatic | **explicit `step.succeed()`** |
| `audit_schema` | `configure()` global | explicit `AUDIT` | **explicit `AUDIT`** |

The only repeated thing left is `except Exception as e: step.fail(e); raise`.
That is deliberate: it is what preserves cell-by-cell execution. You keep the
interactive dev loop (run a cell, inspect `bronze_df`, run the next) and still
lose the boilerplate and the footgun.